In [1]:
from pathlib import Path

MODEL='ETLTC F'

DATASET_PATH=Path('preprocessed_data')

DATASET_PATH.mkdir(parents=True, exist_ok=True)

EVALUATE = True

dataset_list = ['kmnist', 'K49', 'kanjivg', 'kkanji2', 'text_renderer']

VERTICAL_DATASET_PATH = Path('preprocessed_data/text_renderer_vertical')

CHECK_CER = ['text_renderer']

DATASETS_TO_EXCLUDE = ['kmnist', 'K49', 'kanjivg'] #Datasets to exclude from evaluation. If Empty then checks DATASET_TO_CHECK

DATASET_TO_CHECK = ['kkanji2'] #Datasets to show images and example predictions from

In [2]:
import json
def get_word_files(dataset_path, tr=False):
    if tr:
        from pathlib import Path

        with open(str(DATASET_PATH / 'text_renderer' / 'test_labels.json'), 'r') as fp:
            data_dict = json.load(fp)

        tr_dict = dict()

        for items in data_dict.items():
            tr_dict[str(DATASET_PATH / 'text_renderer' / 'images' / items[0]) + '.jpg'] = items[1]


        with open(str(VERTICAL_DATASET_PATH / 'test_labels.json'), 'r') as fp:
            data_dict = json.load(fp)

        vertical_dict = dict()

        for items in data_dict.items():
            vertical_dict[str(VERTICAL_DATASET_PATH / 'images' / items[0]) + '.jpg'] = items[1]

        words_files = list(tr_dict.items()) + list(vertical_dict.items())
    else:
        with open(str(dataset_path / 'test_labels.json'), 'r') as fp:
            data_dict = json.load(fp)
        words_files = list(data_dict.items())
    print(words_files[0])
    print(f"{len(words_files)} words")
    return(words_files)

In [3]:
words_list = []


for dataset in dataset_list:
    if dataset in DATASET_TO_CHECK:
        if dataset == 'text_renderer':
            words_list.append((get_word_files(DATASET_PATH / dataset, True), dataset))
        else:
            words_list.append((get_word_files(DATASET_PATH / dataset), dataset))

('preprocessed_data/kkanji2/1d6a559150683519.png', '間')
13491 words


In [4]:
from typing import Tuple
import tqdm
import torch
from dtrocr.config import DTrOCRConfig
from torch.utils.data import DataLoader
import torch
torch.set_float32_matmul_precision('high')
from dtrocr.model import DTrOCRLMHeadModel


model = DTrOCRLMHeadModel(DTrOCRConfig())
model = torch.compile(model)
model.load_state_dict(torch.load(f'../models/{MODEL}.pt'))
model.to(device=0)



/home/kane/miniconda3/envs/DTrOCR/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OptimizedModule(
  (_orig_mod): DTrOCRLMHeadModel(
    (transformer): DTrOCRModel(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(4, 8), stride=(4, 8))
      )
      (token_embedding): Embedding(32000, 768)
      (positional_embedding): Embedding(256, 768)
      (hidden_layers): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2SdpaAttention(
            (c_attn): Conv1D()
            (c_proj): Conv1D()
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D()
            (c_proj): Conv1D()
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)


In [6]:
import tqdm
import multiprocessing as mp
from util.model import evaluate_model, check_cer
from util.data_processing import get_words_list, WORDSDataset

train_word_records = {}
for pair in words_list:
    if pair[1] not in CHECK_CER and pair[1] in DATASETS_TO_EXCLUDE : continue
    words = get_words_list(pair[0])
    train_word_records[pair[1]] = words
    data = WORDSDataset(words=words, config=DTrOCRConfig())
    dataloader = DataLoader(data, batch_size=32, shuffle=True, num_workers=mp.cpu_count())
    loss, accuracy, cer = 0, 0, 0
    if pair[1] not in DATASETS_TO_EXCLUDE:
        loss, accuracy = evaluate_model(model, dataloader)
    # if pair[1] == 'text_renderer':
    #     cer = check_cer(model, words)    
            
    print(f"{pair[1]} loss: {loss}, accuracy: {accuracy}")

Building dataset: 100%|██████████| 13491/13491 [00:01<00:00, 8597.31it/s]


Word(file_path='preprocessed_data/kkanji2/237599011d7599b9.png', transcription='浄')


/home/kane/miniconda3/envs/DTrOCR/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Evaluating test set: 100%|██████████| 422/422 [01:53<00:00,  3.71it/s]

kkanji2 loss: 0.18170454102056321, accuracy: 0.9643894904315189


In [ ]:
from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig

model.eval()
model.to('cpu')
test_processor = DTrOCRProcessor(DTrOCRConfig())

/home/kane/miniconda3/envs/DTrOCR/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
import matplotlib

matplotlib.rc('font', family='TakaoPGothic')

In [ ]:
# from PIL import Image
# import numpy as np
# import matplotlib.pyplot as plt
# from util.model import test
# from PIL import Image

# print(train_word_records[DATASET_TO_CHECK[0]])

# for ds in DATASET_TO_CHECK:
#     test(model, train_word_records[ds], 5)

In [ ]:
tokeniser = test_processor.tokeniser
print(tokeniser.pad_token)
print(tokeniser.eos_token)
print(tokeniser.bos_token)
print(tokeniser.model_max_length)
print(tokeniser.sep_token_id)

print(tokeniser('<s>[SEP]'))

[PAD]
</s>
<s>
128
5
{'input_ids': [1, 5, 2], 'attention_mask': [1, 1, 1]}
